# Task 2 — Wikispeedia Next-Click Prediction (Inference Notebook)

**Data setup.** The competition tables are provided by the Datathon Task 2 Kaggle competition: https://www.kaggle.com/competitions/datathon-task-2. The article link graph is reconstructed
(blue-link OCR) in our companion **build notebook**; its output `graph_edges.parquet` is attached
here as a Kaggle Dataset.

**Model.** Candidates = the current page's real out-links. We keep the candidate(s) at minimum
graph-distance to the target (the human "get closer" rule), then break ties with a fixed scorer:

```
tiebreak(a) = 1.0*PageRank(a) + 8.0*sim(a,t) + 4.0*flow(a,t)
            + 0.5*catpop(a,t) + 1.0*log_indeg(a) + 0.5*cat_match(a,t)
```

- `PageRank(a)` — global hubness on the reconstructed graph (denser than train popularity)
- `sim(a,t)` — TF-IDF cosine (title+categories) to the target
- `flow(a,t)` — fraction of a's out-links that move strictly closer to the target
- `catpop(a,t)` — how often `a` is the click when heading to the target's category (from train)
- `log_indeg`, `cat_match` — in-degree hubness and shared top-category

Fresh-seed holdout ≈ **0.74**. No learned parameters → *Run All* reproduces the submission exactly.


In [1]:
import os, glob
import numpy as np, pandas as pd
from collections import defaultdict, deque, Counter
from sklearn.feature_extraction.text import TfidfVectorizer
import scipy.sparse as sp
import networkx as nx

def _find(name): return sorted(glob.glob(f'/kaggle/input/**/{name}', recursive=True))
CANDS = _find('states_test.csv')
assert CANDS, 'Competition data not attached. /kaggle/input has: ' + str(os.listdir('/kaggle/input'))
DIR = os.path.dirname(CANDS[0]); print('data:', DIR)
train=pd.read_csv(f'{DIR}/states_train.csv'); test=pd.read_csv(f'{DIR}/states_test.csv')
articles=pd.read_csv(f'{DIR}/articles.csv'); cats=pd.read_csv(f'{DIR}/categories.csv')
GP = _find('graph_edges.parquet')
assert GP, 'graph_edges.parquet not found — add your wikispeedia-graph dataset as Input'
edges=pd.read_parquet(GP[0]); print('edges:', len(edges))
N=int(articles.article_id.max())+1

data: /kaggle/input/competitions/datathon-task-2/dataset-task2
edges: 93830


In [2]:
# PageRank on the OCR graph only (label-independent, dense hubness)
Gd=nx.DiGraph(); Gd.add_edges_from((int(s),int(d)) for s,d in zip(edges.src,edges.dst))
PR=nx.pagerank(Gd, alpha=0.85)

# adjacency = OCR edges + observed train transitions (both are real links)
out_links=defaultdict(set); in_links=defaultdict(set)
for s,d in zip(edges.src,edges.dst): out_links[int(s)].add(int(d)); in_links[int(d)].add(int(s))
for c,n in zip(train.current_article_id,train.next_article_id): out_links[c].add(n); in_links[n].add(c)

gpop=Counter(train.next_article_id); GT=gpop.most_common(1)[0][0]
TOPHUBS=[a for a,_ in gpop.most_common(30)]

# top-level category per article; target-category-conditioned next popularity (from train)
cats['top']=cats.category.str.split('.').str[1]
topcat={}
for a,tp in zip(cats.article_id,cats.top):
    if a not in topcat: topcat[a]=tp
catpop=defaultdict(Counter)
for tt,nn in zip(train.target_article_id,train.next_article_id): catpop[topcat.get(tt)][nn]+=1

# TF-IDF (title + category tokens)
cat_by=defaultdict(list)
for a,c in zip(cats.article_id,cats.category): cat_by[a].append(c)
title=dict(zip(articles.article_id,articles.title))
def art_text(a):
    toks=[str(title.get(a,"")).replace("-"," ").replace(","," ")]
    for c in cat_by.get(a,[]): toks.append(c.replace("subject.","").replace("."," ").replace("_"," "))
    return " ".join(toks)
Xn=TfidfVectorizer().fit_transform([art_text(a) for a in range(N)])
Xn=sp.csr_matrix(Xn.multiply(1.0/(np.sqrt(Xn.multiply(Xn).sum(1))+1e-9)))

def bfs_dist_to(t):
    dist={t:0}; q=deque([t])
    while q:
        u=q.popleft()
        for v in in_links.get(u,()):
            if v not in dist: dist[v]=dist[u]+1; q.append(v)
    return dist
def cand_set(c):
    s=set(out_links.get(c,set()))
    if len(s)<2: s|=set(TOPHUBS)
    return s

In [3]:
W_PR,W_SIM,W_FLOW,W_CP,W_IND,W_CM = 1.0,8.0,4.0,0.5,1.0,0.5
dist_cache={}; preds=[]
for c,t in zip(test.current_article_id, test.target_article_id):
    cand=list(cand_set(c))
    if not cand: preds.append(GT); continue
    if t not in dist_cache: dist_cache[t]=bfs_dist_to(t)
    dist=dist_cache[t]
    cand=np.asarray(cand)
    d_a=np.array([dist.get(a,30) for a in cand],float)
    pr =np.log(np.array([PR.get(int(a),1e-8) for a in cand],float)+1e-12)
    simt=np.asarray(Xn[cand].dot(Xn[t].T).todense()).ravel()
    ind=np.log1p(np.array([len(in_links.get(a,())) for a in cand],float))
    cc=catpop.get(topcat.get(t),{})
    cp=np.log1p(np.array([cc.get(a,0) for a in cand],float))
    tct=topcat.get(t)
    cm=np.array([1.0 if topcat.get(a)==tct else 0.0 for a in cand])
    flow=np.zeros(len(cand))
    for i,a in enumerate(cand):
        nb=out_links.get(a)
        if nb: flow[i]=sum(1 for v in nb if dist.get(v,30)<d_a[i])/len(nb)
    sc=W_PR*pr + W_SIM*simt + W_FLOW*flow + W_CP*cp + W_IND*ind + W_CM*cm
    sc=np.where(d_a<=d_a.min(), sc, -1e9)         # strict min-distance filter
    preds.append(int(cand[int(np.argmax(sc))]))

sub=pd.DataFrame({"state_id":test.state_id,"predicted_next_article_id":preds})
sub.to_csv("/kaggle/working/submission.csv", index=False)
print("rows:",len(sub),"unique:",sub.predicted_next_article_id.nunique()); print(sub.head())

rows: 6000 unique: 736
   state_id  predicted_next_article_id
0         0                       4393
1         6                       1808
2         8                        237
3        10                       1689
4        11                       4503
